In [1]:
print("OK")

OK


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]

In [3]:
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_azure_ai.utils.agents import pretty_print

model = AzureAIOpenAIApiChatModel(
    endpoint="https://grashikreddy-8785-resource.services.ai.azure.com/openai/v1/",
    credential=azure_openai_key, #if using Entra ID you can should use DefaultAzureCredential() instead
    model="gpt-5.4-mini",
    temperature=0.1
)

In [4]:
from langchain_azure_ai.embeddings import AzureAIOpenAIApiEmbeddingsModel

embedding_model = AzureAIOpenAIApiEmbeddingsModel(
    endpoint="https://grashikreddy-8785-resource.services.ai.azure.com/openai/v1/",
    credential=azure_openai_key, #if using Entra ID you can should use DefaultAzureCredential() instead
    model="text-embedding-3-small"
)

In [8]:
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic import hub

# Load page content from a URL
url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
response = requests.get(url, headers={"User-Agent": "rag-demo/1.0"}, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
for tag in soup(["script", "style", "nav", "header", "footer"]):
    tag.decompose()
page_text = soup.get_text(separator="\n", strip=True)

title = soup.title.string if soup.title else ""
docs = [Document(page_content=page_text, metadata={"source": url, "title": title})]
print(f"Loaded {len(page_text)} characters from {url}")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

# Embed chunks and store in Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="rag_demo",
    persist_directory="./chroma_db",
)

Loaded 22907 characters from https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Split into 30 chunks


In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# RAG prompt: answer only from the retrieved context
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. "
     "Use only the following retrieved context to answer the question. "
     "If the answer is not in the context, say you don't know. "
     "Use three sentences maximum and keep the answer concise.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

answer = rag_chain.invoke("RAGA RAG")
print(answer)

RAG improves LLMs by adding information retrieval before generating responses, pulling relevant text from databases, uploaded documents, or web sources. It can also reduce hallucinations, save retraining costs, and let users verify cited sources.
